# Fractional Factorial Analysis for Neuro-Symbolic Pipeline

This notebook analyzes results from a $2^{4-1}_{IV}$ fractional factorial experiment evaluating a neuro-symbolic reasoning pipeline.

## Experimental Design

**Factors:**
- **T1 (`use_openie`)**: Use Stanford CoreNLP OpenIE relation triples during text-to-logic conversion
- **T2 (`use_enrichment_kb`)**: Apply enrichment (modal-pair verification, negation detection, finite-domain auxiliaries, conflict resolution)
- **Q1 (`use_shortcuts`)**: Activate deterministic shortcut detectors (modal opposites, lexical antonyms, implication contradictions)
- **Q2 (`expand_query`)**: Create query alternatives using WordNet synonyms + majority-vote ensembling

**Generator**: $T_2 = Q_1 \cdot Q_2 \cdot T_1$

**Datasets**: LogiQA2, LogicBench, DocNLI, Alice

**Response Variables**:
- Overall accuracy
- Per-category accuracy: Entailment, Contradiction, Uncertain, Not Mentioned

In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas numpy scipy statsmodels matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

# Define constants
DATASETS = ['LogiQA2', 'LogicBench', 'DocNLI', 'Alice']
CATEGORIES = ['entailment', 'contradiction', 'uncertain', 'not_mentioned']
FACTORS = ['Q1_shortcuts', 'Q2_expand', 'T1_openie', 'T2_enrich']
FACTOR_LABELS = ['Q1: Shortcuts', 'Q2: Expand Query', 'T1: OpenIE', 'T2: Enrich KB']

## 1. Define the Fractional Factorial Design

The $2^{4-1}_{IV}$ design with generator $T_2 = Q_1 \cdot Q_2 \cdot T_1$ has resolution IV, meaning:
- All main effects are estimable and unconfounded with each other
- Main effects are confounded only with 3-factor interactions
- Two-factor interactions are confounded with other two-factor interactions

**Aliasing structure** (defining relation $I = Q_1 Q_2 T_1 T_2$):
- $Q_1 \cdot Q_2$ is aliased with $T_1 \cdot T_2$
- $Q_1 \cdot T_1$ is aliased with $Q_2 \cdot T_2$
- $Q_1 \cdot T_2$ is aliased with $Q_2 \cdot T_1$

In [ ]:
# Define the 2^{4-1} fractional factorial design
# Generator: T2 = Q1 * Q2 * T1
# Using -1 for low level, +1 for high level

design = pd.DataFrame({
    'Run': [1, 2, 3, 4, 5, 6, 7, 8],
    'Q1_shortcuts': [-1, +1, -1, +1, -1, +1, -1, +1],
    'Q2_expand': [-1, -1, +1, +1, -1, -1, +1, +1],
    'T1_openie': [-1, -1, -1, -1, +1, +1, +1, +1],
})

# T2 = Q1 * Q2 * T1 (generator)
design['T2_enrich'] = design['Q1_shortcuts'] * design['Q2_expand'] * design['T1_openie']

# Verify the design matches the paper's Table
print("Fractional Factorial Design (2^{4-1}_IV):")
print("="*60)
design_display = design.copy()
design_display.columns = ['Run', 'Q1:Shortcuts', 'Q2:Expand', 'T1:OpenIE', 'T2:Enrich']
# Convert -1/+1 to -/+ for display
for col in ['Q1:Shortcuts', 'Q2:Expand', 'T1:OpenIE', 'T2:Enrich']:
    design_display[col] = design_display[col].map({-1: '-', 1: '+'})
print(design_display.to_string(index=False))

In [ ]:
# Verify aliasing structure
print("\nAliasing Structure Verification:")
print("="*60)
print(f"Defining relation I = Q1*Q2*T1*T2: {(design['Q1_shortcuts'] * design['Q2_expand'] * design['T1_openie'] * design['T2_enrich']).unique()}")
print("\nAliased pairs (confounded 2-factor interactions):")
print(f"  Q1*Q2 aliased with T1*T2: {np.allclose(design['Q1_shortcuts']*design['Q2_expand'], design['T1_openie']*design['T2_enrich'])}")
print(f"  Q1*T1 aliased with Q2*T2: {np.allclose(design['Q1_shortcuts']*design['T1_openie'], design['Q2_expand']*design['T2_enrich'])}")
print(f"  Q1*T2 aliased with Q2*T1: {np.allclose(design['Q1_shortcuts']*design['T2_enrich'], design['Q2_expand']*design['T1_openie'])}")

---
## 2. Input Your Experimental Results

**Instructions**: Replace the placeholder values below with your actual accuracy results.

For each dataset and each run, enter:
- **Overall accuracy** (or leave as 0 to auto-compute from categories)
- **Per-category accuracy**: entailment, contradiction, uncertain, not_mentioned

Values can be 0-1 (proportions) or 0-100 (percentages) - the code handles both.

In [ ]:
# ============================================================================
# INPUT YOUR RESULTS HERE
# ============================================================================
# For each dataset, enter results for all 8 runs.
# Each run needs: overall accuracy AND per-category accuracies
# 
# Run configurations:
#   Run 1: Q1-, Q2-, T1-, T2-
#   Run 2: Q1+, Q2-, T1-, T2+
#   Run 3: Q1-, Q2+, T1-, T2+
#   Run 4: Q1+, Q2+, T1-, T2-
#   Run 5: Q1-, Q2-, T1+, T2+
#   Run 6: Q1+, Q2-, T1+, T2-
#   Run 7: Q1-, Q2+, T1+, T2-
#   Run 8: Q1+, Q2+, T1+, T2+
# ============================================================================

# LogiQA2 Results
logiqa2_results = {
    'Run': [1, 2, 3, 4, 5, 6, 7, 8],
    'overall':        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'entailment':     [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'contradiction':  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'uncertain':      [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'not_mentioned':  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
}

# LogicBench Results
logicbench_results = {
    'Run': [1, 2, 3, 4, 5, 6, 7, 8],
    'overall':        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'entailment':     [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'contradiction':  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'uncertain':      [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'not_mentioned':  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
}

# DocNLI Results
docnli_results = {
    'Run': [1, 2, 3, 4, 5, 6, 7, 8],
    'overall':        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'entailment':     [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'contradiction':  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'uncertain':      [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'not_mentioned':  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
}

# Alice Results
alice_results = {
    'Run': [1, 2, 3, 4, 5, 6, 7, 8],
    'overall':        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'entailment':     [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'contradiction':  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'uncertain':      [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'not_mentioned':  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
}

In [ ]:
# ============================================================================
# Process and combine all results
# ============================================================================

def process_dataset_results(results_dict, dataset_name):
    """Process a single dataset's results and normalize to 0-1 scale."""
    df = pd.DataFrame(results_dict)
    
    # Convert to 0-1 scale if needed
    value_cols = ['overall', 'entailment', 'contradiction', 'uncertain', 'not_mentioned']
    for col in value_cols:
        if df[col].max() > 1:
            df[col] = df[col] / 100.0
    
    # Auto-compute overall if it's all zeros but categories have values
    if df['overall'].sum() == 0 and df[CATEGORIES].sum().sum() > 0:
        df['overall'] = df[CATEGORIES].mean(axis=1)
        print(f"  {dataset_name}: Auto-computed overall accuracy from category means")
    
    # Rename columns to include dataset prefix
    df = df.rename(columns={
        'overall': f'{dataset_name}_overall',
        'entailment': f'{dataset_name}_entailment',
        'contradiction': f'{dataset_name}_contradiction',
        'uncertain': f'{dataset_name}_uncertain',
        'not_mentioned': f'{dataset_name}_not_mentioned',
    })
    
    return df

# Process each dataset
print("Processing results...")
logiqa2_df = process_dataset_results(logiqa2_results, 'LogiQA2')
logicbench_df = process_dataset_results(logicbench_results, 'LogicBench')
docnli_df = process_dataset_results(docnli_results, 'DocNLI')
alice_df = process_dataset_results(alice_results, 'Alice')

# Merge all results with design
df = design.copy()
for results_df in [logiqa2_df, logicbench_df, docnli_df, alice_df]:
    df = df.merge(results_df, on='Run')

# Create aggregate columns
overall_cols = [f'{ds}_overall' for ds in DATASETS]
df['Average_overall'] = df[overall_cols].mean(axis=1)

for cat in CATEGORIES:
    cat_cols = [f'{ds}_{cat}' for ds in DATASETS]
    df[f'Average_{cat}'] = df[cat_cols].mean(axis=1)

print("\nData loaded successfully!")
print(f"Shape: {df.shape}")

In [ ]:
# Display summary of input data
print("\n" + "="*80)
print("INPUT DATA SUMMARY")
print("="*80)

for dataset in DATASETS:
    print(f"\n{dataset}:")
    cols = [f'{dataset}_overall'] + [f'{dataset}_{cat}' for cat in CATEGORIES]
    display_df = df[['Run'] + cols].copy()
    display_df.columns = ['Run', 'Overall'] + [cat.title() for cat in CATEGORIES]
    print(display_df.to_string(index=False))

---
## 3. Core Analysis Functions

In [ ]:
def compute_effects(df, response_col):
    """
    Compute main effects and two-factor interactions for a 2^{4-1} design.
    
    For a 2-level design, the effect of factor A is:
        Effect(A) = mean(Y | A=+1) - mean(Y | A=-1)
    """
    effects = {}
    
    # Main effects
    for factor in FACTORS:
        high = df[df[factor] == 1][response_col].mean()
        low = df[df[factor] == -1][response_col].mean()
        effects[factor] = high - low
    
    # Two-factor interactions (with aliasing noted)
    df_temp = df.copy()
    
    # Q1*Q2 (aliased with T1*T2)
    df_temp['Q1_Q2'] = df_temp['Q1_shortcuts'] * df_temp['Q2_expand']
    effects['Q1*Q2 (=T1*T2)'] = df_temp[df_temp['Q1_Q2'] == 1][response_col].mean() - df_temp[df_temp['Q1_Q2'] == -1][response_col].mean()
    
    # Q1*T1 (aliased with Q2*T2)
    df_temp['Q1_T1'] = df_temp['Q1_shortcuts'] * df_temp['T1_openie']
    effects['Q1*T1 (=Q2*T2)'] = df_temp[df_temp['Q1_T1'] == 1][response_col].mean() - df_temp[df_temp['Q1_T1'] == -1][response_col].mean()
    
    # Q1*T2 (aliased with Q2*T1)
    df_temp['Q1_T2'] = df_temp['Q1_shortcuts'] * df_temp['T2_enrich']
    effects['Q1*T2 (=Q2*T1)'] = df_temp[df_temp['Q1_T2'] == 1][response_col].mean() - df_temp[df_temp['Q1_T2'] == -1][response_col].mean()
    
    return effects


def compute_effect_significance(df, response_col, n_bootstrap=10000):
    """
    Use bootstrap to estimate confidence intervals for effects.
    """
    np.random.seed(42)
    original_effects = compute_effects(df.copy(), response_col)
    
    bootstrap_effects = {key: [] for key in original_effects.keys()}
    
    for _ in range(n_bootstrap):
        boot_df = df.sample(n=len(df), replace=True)
        boot_effects = compute_effects(boot_df.copy(), response_col)
        for key in bootstrap_effects:
            bootstrap_effects[key].append(boot_effects[key])
    
    results = []
    for key in original_effects:
        boot_array = np.array(bootstrap_effects[key])
        ci_low = np.percentile(boot_array, 2.5)
        ci_high = np.percentile(boot_array, 97.5)
        
        if original_effects[key] > 0:
            p_value = np.mean(boot_array <= 0) * 2
        else:
            p_value = np.mean(boot_array >= 0) * 2
        p_value = min(p_value, 1.0)
        
        results.append({
            'Effect': key,
            'Estimate': original_effects[key],
            'CI_Low': ci_low,
            'CI_High': ci_high,
            'p_value': p_value,
            'Significant': 'Yes' if p_value < 0.05 else 'No'
        })
    
    return pd.DataFrame(results)


def display_effects(effects, title):
    """Display effects in a formatted table."""
    print(f"\n{'='*65}")
    print(f"Effects for {title}")
    print(f"{'='*65}")
    print(f"{'Effect':<25} {'Estimate':>12} {'Magnitude':>15} {'Direction':>10}")
    print("-"*65)
    for name, value in effects.items():
        direction = '+' if value > 0 else '-' if value < 0 else '0'
        if abs(value) > 0.10:
            magnitude = 'Strong'
        elif abs(value) > 0.05:
            magnitude = 'Moderate'
        else:
            magnitude = 'Weak'
        print(f"{name:<25} {value:>+12.4f} {magnitude:>15} {direction:>10}")
    return effects

---
## 4. Overall Accuracy Analysis

In [ ]:
# Compute effects for overall accuracy (each dataset)
print("\n" + "#"*80)
print("# OVERALL ACCURACY ANALYSIS")
print("#"*80)

overall_effects = {}
for dataset in DATASETS:
    col = f'{dataset}_overall'
    effects = compute_effects(df.copy(), col)
    display_effects(effects, f'{dataset} (Overall)')
    overall_effects[dataset] = effects

# Average across datasets
avg_overall_effects = compute_effects(df.copy(), 'Average_overall')
display_effects(avg_overall_effects, 'AVERAGE Overall (All Datasets)')

In [ ]:
# Statistical significance for overall accuracy
print("\n" + "="*80)
print("BOOTSTRAP SIGNIFICANCE TESTING - Overall Accuracy")
print("="*80)

overall_significance = {}
for dataset in DATASETS + ['Average']:
    col = f'{dataset}_overall'
    print(f"\n{dataset}:")
    print("-"*80)
    sig_df = compute_effect_significance(df.copy(), col)
    overall_significance[dataset] = sig_df
    print(sig_df.to_string(index=False))

---
## 5. Per-Category Accuracy Analysis

In [ ]:
# Compute effects for each category
print("\n" + "#"*80)
print("# PER-CATEGORY ACCURACY ANALYSIS")
print("#"*80)

category_effects = {cat: {} for cat in CATEGORIES}

for cat in CATEGORIES:
    print(f"\n{'='*80}")
    print(f"CATEGORY: {cat.upper()}")
    print(f"{'='*80}")
    
    for dataset in DATASETS:
        col = f'{dataset}_{cat}'
        effects = compute_effects(df.copy(), col)
        display_effects(effects, f'{dataset} ({cat.title()})')
        category_effects[cat][dataset] = effects
    
    # Average across datasets for this category
    avg_col = f'Average_{cat}'
    avg_effects = compute_effects(df.copy(), avg_col)
    display_effects(avg_effects, f'AVERAGE {cat.title()} (All Datasets)')
    category_effects[cat]['Average'] = avg_effects

In [ ]:
# Statistical significance for each category (averaged across datasets)
print("\n" + "="*80)
print("BOOTSTRAP SIGNIFICANCE TESTING - Per Category (Averaged)")
print("="*80)

category_significance = {}
for cat in CATEGORIES:
    col = f'Average_{cat}'
    print(f"\n{cat.upper()}:")
    print("-"*80)
    sig_df = compute_effect_significance(df.copy(), col)
    category_significance[cat] = sig_df
    print(sig_df.to_string(index=False))

---
## 6. Visualizations

In [ ]:
# Main Effects Plot - Overall Accuracy by Dataset
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, dataset in enumerate(DATASETS):
    ax = axes[idx // 2, idx % 2]
    
    x_positions = np.arange(len(FACTORS))
    effects = [overall_effects[dataset][f] for f in FACTORS]
    colors = ['green' if e > 0 else 'red' for e in effects]
    
    bars = ax.bar(x_positions, effects, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(FACTOR_LABELS, rotation=45, ha='right')
    ax.set_ylabel('Effect on Accuracy')
    ax.set_title(f'{dataset}: Main Effects (Overall)')
    ax.set_ylim(-0.4, 0.4)
    
    for bar, effect in zip(bars, effects):
        height = bar.get_height()
        ax.annotate(f'{effect:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3 if height >= 0 else -12),
                    textcoords="offset points",
                    ha='center', va='bottom' if height >= 0 else 'top',
                    fontsize=9)

plt.tight_layout()
plt.savefig('main_effects_overall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Main Effects by Category (averaged across datasets)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, cat in enumerate(CATEGORIES):
    ax = axes[idx // 2, idx % 2]
    
    x_positions = np.arange(len(FACTORS))
    effects = [category_effects[cat]['Average'][f] for f in FACTORS]
    colors = ['green' if e > 0 else 'red' for e in effects]
    
    bars = ax.bar(x_positions, effects, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(FACTOR_LABELS, rotation=45, ha='right')
    ax.set_ylabel('Effect on Accuracy')
    ax.set_title(f'{cat.title()}: Main Effects (Avg across datasets)')
    ax.set_ylim(-0.4, 0.4)
    
    for bar, effect in zip(bars, effects):
        height = bar.get_height()
        ax.annotate(f'{effect:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3 if height >= 0 else -12),
                    textcoords="offset points",
                    ha='center', va='bottom' if height >= 0 else 'top',
                    fontsize=9)

plt.tight_layout()
plt.savefig('main_effects_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Effects by Dataset (Overall Accuracy)
fig, ax = plt.subplots(figsize=(12, 5))

effect_matrix = pd.DataFrame(overall_effects).T
effect_matrix = effect_matrix[FACTORS + ['Q1*Q2 (=T1*T2)', 'Q1*T1 (=Q2*T2)', 'Q1*T2 (=Q2*T1)']]

sns.heatmap(effect_matrix, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            vmin=-0.3, vmax=0.3, ax=ax, cbar_kws={'label': 'Effect on Accuracy'})
ax.set_title('Effect Heatmap: Overall Accuracy by Dataset')
ax.set_xlabel('Effect')
ax.set_ylabel('Dataset')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('heatmap_overall_by_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Effects by Category (Averaged across datasets)
fig, ax = plt.subplots(figsize=(12, 5))

cat_effect_matrix = {}
for cat in CATEGORIES:
    cat_effect_matrix[cat.title()] = category_effects[cat]['Average']

effect_matrix = pd.DataFrame(cat_effect_matrix).T
effect_matrix = effect_matrix[FACTORS + ['Q1*Q2 (=T1*T2)', 'Q1*T1 (=Q2*T2)', 'Q1*T2 (=Q2*T1)']]

sns.heatmap(effect_matrix, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            vmin=-0.3, vmax=0.3, ax=ax, cbar_kws={'label': 'Effect on Accuracy'})
ax.set_title('Effect Heatmap: By Category (Averaged Across Datasets)')
ax.set_xlabel('Effect')
ax.set_ylabel('Category')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('heatmap_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comprehensive Heatmap: All Dataset x Category combinations
fig, ax = plt.subplots(figsize=(14, 10))

# Build a matrix with rows = (Dataset, Category) and cols = Factors
all_effects_data = {}
for dataset in DATASETS:
    # Overall
    all_effects_data[f'{dataset} (Overall)'] = overall_effects[dataset]
    # Per category
    for cat in CATEGORIES:
        all_effects_data[f'{dataset} ({cat.title()})'] = category_effects[cat][dataset]

full_matrix = pd.DataFrame(all_effects_data).T
full_matrix = full_matrix[FACTORS]  # Just main effects for clarity

sns.heatmap(full_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-0.4, vmax=0.4, ax=ax, cbar_kws={'label': 'Effect'})
ax.set_title('Complete Effect Heatmap: All Datasets × Categories')
ax.set_xlabel('Factor')
ax.set_ylabel('Dataset (Category)')

plt.tight_layout()
plt.savefig('heatmap_complete.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pareto Chart - Average Overall Effects
fig, ax = plt.subplots(figsize=(10, 6))

effect_names = list(avg_overall_effects.keys())
effect_values = [abs(avg_overall_effects[e]) for e in effect_names]
effect_signs = ['+' if avg_overall_effects[e] > 0 else '-' for e in effect_names]

sorted_idx = np.argsort(effect_values)[::-1]
sorted_names = [effect_names[i] for i in sorted_idx]
sorted_values = [effect_values[i] for i in sorted_idx]
sorted_signs = [effect_signs[i] for i in sorted_idx]
sorted_colors = ['green' if s == '+' else 'red' for s in sorted_signs]

bars = ax.barh(range(len(sorted_names)), sorted_values, color=sorted_colors, alpha=0.7, edgecolor='black')
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels([f"{n} ({s})" for n, s in zip(sorted_names, sorted_signs)])
ax.set_xlabel('|Effect| on Accuracy')
ax.set_title('Pareto Chart: Effect Magnitudes (Average Overall Accuracy)')
ax.invert_yaxis()

ax.axvline(x=0.05, color='orange', linestyle='--', linewidth=2, label='Moderate (5%)')
ax.axvline(x=0.10, color='red', linestyle='--', linewidth=2, label='Strong (10%)')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('pareto_effects.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Interaction Plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

response = 'Average_overall'

# Q1 x Q2 interaction
ax = axes[0]
for q1_level in [-1, 1]:
    subset = df[df['Q1_shortcuts'] == q1_level]
    means = subset.groupby('Q2_expand')[response].mean()
    label = 'Q1=+' if q1_level == 1 else 'Q1=-'
    ax.plot([-1, 1], [means.get(-1, 0), means.get(1, 0)], 'o-', label=label, markersize=8)
ax.set_xticks([-1, 1])
ax.set_xticklabels(['Q2=-', 'Q2=+'])
ax.set_xlabel('Q2: Expand Query')
ax.set_ylabel('Average Accuracy')
ax.set_title('Q1 x Q2 Interaction\n(aliased with T1 x T2)')
ax.legend()

# Q1 x T1 interaction
ax = axes[1]
for q1_level in [-1, 1]:
    subset = df[df['Q1_shortcuts'] == q1_level]
    means = subset.groupby('T1_openie')[response].mean()
    label = 'Q1=+' if q1_level == 1 else 'Q1=-'
    ax.plot([-1, 1], [means.get(-1, 0), means.get(1, 0)], 'o-', label=label, markersize=8)
ax.set_xticks([-1, 1])
ax.set_xticklabels(['T1=-', 'T1=+'])
ax.set_xlabel('T1: OpenIE')
ax.set_ylabel('Average Accuracy')
ax.set_title('Q1 x T1 Interaction\n(aliased with Q2 x T2)')
ax.legend()

# T1 x T2 interaction
ax = axes[2]
for t1_level in [-1, 1]:
    subset = df[df['T1_openie'] == t1_level]
    means = subset.groupby('T2_enrich')[response].mean()
    label = 'T1=+' if t1_level == 1 else 'T1=-'
    ax.plot([-1, 1], [means.get(-1, 0), means.get(1, 0)], 'o-', label=label, markersize=8)
ax.set_xticks([-1, 1])
ax.set_xticklabels(['T2=-', 'T2=+'])
ax.set_xlabel('T2: Enrich KB')
ax.set_ylabel('Average Accuracy')
ax.set_title('T1 x T2 Interaction\n(aliased with Q1 x Q2)')
ax.legend()

plt.tight_layout()
plt.savefig('interaction_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Category comparison by configuration
fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for run_idx in range(8):
    ax = axes[run_idx // 4, run_idx % 4]
    row = df.iloc[run_idx]
    
    # Get category accuracies for each dataset
    x = np.arange(len(DATASETS))
    width = 0.2
    
    for i, cat in enumerate(CATEGORIES):
        values = [row[f'{ds}_{cat}'] for ds in DATASETS]
        ax.bar(x + i*width - 1.5*width, values, width, label=cat.title(), alpha=0.8)
    
    ax.set_ylabel('Accuracy')
    ax.set_title(f"Run {int(row['Run'])}: Q1{'+' if row['Q1_shortcuts']==1 else '-'} "
                 f"Q2{'+' if row['Q2_expand']==1 else '-'} "
                 f"T1{'+' if row['T1_openie']==1 else '-'} "
                 f"T2{'+' if row['T2_enrich']==1 else '-'}")
    ax.set_xticks(x)
    ax.set_xticklabels(DATASETS, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 1)
    if run_idx == 0:
        ax.legend(loc='upper right', fontsize=7)

plt.tight_layout()
plt.savefig('category_by_config.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Summary Statistics

In [ ]:
# Best and worst configurations
print("\n" + "="*80)
print("PERFORMANCE SUMMARY - Overall Accuracy")
print("="*80)

for dataset in DATASETS + ['Average']:
    col = f'{dataset}_overall'
    best_idx = df[col].idxmax()
    worst_idx = df[col].idxmin()
    
    print(f"\n{dataset}:")
    print(f"  Mean: {df[col].mean():.3f}, Std: {df[col].std():.3f}")
    print(f"  Best:  Run {df.loc[best_idx, 'Run']:.0f} (Acc={df.loc[best_idx, col]:.3f}) - "
          f"Q1{'+' if df.loc[best_idx, 'Q1_shortcuts']==1 else '-'}, "
          f"Q2{'+' if df.loc[best_idx, 'Q2_expand']==1 else '-'}, "
          f"T1{'+' if df.loc[best_idx, 'T1_openie']==1 else '-'}, "
          f"T2{'+' if df.loc[best_idx, 'T2_enrich']==1 else '-'}")
    print(f"  Worst: Run {df.loc[worst_idx, 'Run']:.0f} (Acc={df.loc[worst_idx, col]:.3f}) - "
          f"Q1{'+' if df.loc[worst_idx, 'Q1_shortcuts']==1 else '-'}, "
          f"Q2{'+' if df.loc[worst_idx, 'Q2_expand']==1 else '-'}, "
          f"T1{'+' if df.loc[worst_idx, 'T1_openie']==1 else '-'}, "
          f"T2{'+' if df.loc[worst_idx, 'T2_enrich']==1 else '-'}")

In [ ]:
# Category difficulty analysis
print("\n" + "="*80)
print("CATEGORY DIFFICULTY (lower mean = harder)")
print("="*80)

cat_means = {}
for cat in CATEGORIES:
    cat_means[cat] = df[f'Average_{cat}'].mean()

sorted_cats = sorted(cat_means.items(), key=lambda x: x[1])
for i, (cat, mean) in enumerate(sorted_cats, 1):
    print(f"  {i}. {cat.title()}: {mean:.3f}")

In [ ]:
# Dataset difficulty analysis
print("\n" + "="*80)
print("DATASET DIFFICULTY (lower mean = harder)")
print("="*80)

ds_means = {}
for ds in DATASETS:
    ds_means[ds] = df[f'{ds}_overall'].mean()

sorted_ds = sorted(ds_means.items(), key=lambda x: x[1])
for i, (ds, mean) in enumerate(sorted_ds, 1):
    print(f"  {i}. {ds}: {mean:.3f}")

---
## 8. LaTeX Tables for Paper

In [ ]:
def generate_latex_results_table():
    """Generate LaTeX table: Overall accuracy by configuration."""
    print("% LaTeX Table: Overall Accuracy by Configuration")
    print("\\begin{table}[t]")
    print("\\centering")
    print("\\caption{Overall accuracy (\\%) by configuration across datasets.}")
    print("\\label{tab:results_overall}")
    print("\\begin{tabular}{c cccc | cccc | c}")
    print("\\toprule")
    print("Run & $Q_1$ & $Q_2$ & $T_1$ & $T_2$ & LogiQA2 & LogicBench & DocNLI & Alice & Avg \\\\")
    print("\\midrule")
    
    for _, row in df.iterrows():
        q1 = '+' if row['Q1_shortcuts'] == 1 else '-'
        q2 = '+' if row['Q2_expand'] == 1 else '-'
        t1 = '+' if row['T1_openie'] == 1 else '-'
        t2 = '+' if row['T2_enrich'] == 1 else '-'
        
        print(f"{int(row['Run'])} & ${q1}$ & ${q2}$ & ${t1}$ & ${t2}$ & "
              f"{row['LogiQA2_overall']*100:.1f} & {row['LogicBench_overall']*100:.1f} & "
              f"{row['DocNLI_overall']*100:.1f} & {row['Alice_overall']*100:.1f} & "
              f"{row['Average_overall']*100:.1f} \\\\")
    
    print("\\bottomrule")
    print("\\end{tabular}")
    print("\\end{table}")

generate_latex_results_table()

In [ ]:
def generate_latex_effects_table():
    """Generate LaTeX table: Main effects on overall accuracy."""
    print("\n% LaTeX Table: Main Effects on Overall Accuracy")
    print("\\begin{table}[t]")
    print("\\centering")
    print("\\caption{Estimated main effects on overall accuracy (percentage points). Bold indicates $|\\text{effect}| > 5\\%$.}")
    print("\\label{tab:effects_overall}")
    print("\\begin{tabular}{l cccc c}")
    print("\\toprule")
    print("Factor & LogiQA2 & LogicBench & DocNLI & Alice & Average \\\\")
    print("\\midrule")
    
    factor_names_latex = ['$Q_1$: Shortcuts', '$Q_2$: Expand', '$T_1$: OpenIE', '$T_2$: Enrich']
    
    for factor, name in zip(FACTORS, factor_names_latex):
        vals = [overall_effects[ds][factor] for ds in DATASETS]
        avg_val = avg_overall_effects[factor]
        
        formatted = []
        for v in vals + [avg_val]:
            if abs(v) > 0.05:
                formatted.append(f"\\textbf{{{v*100:+.1f}}}")
            else:
                formatted.append(f"{v*100:+.1f}")
        
        print(f"{name} & {' & '.join(formatted)} \\\\")
    
    print("\\bottomrule")
    print("\\end{tabular}")
    print("\\end{table}")

generate_latex_effects_table()

In [ ]:
def generate_latex_category_effects_table():
    """Generate LaTeX table: Main effects by category (averaged across datasets)."""
    print("\n% LaTeX Table: Main Effects by Category")
    print("\\begin{table}[t]")
    print("\\centering")
    print("\\caption{Main effects (percentage points) by response category, averaged across datasets. Bold indicates $|\\text{effect}| > 5\\%$.}")
    print("\\label{tab:effects_category}")
    print("\\begin{tabular}{l cccc}")
    print("\\toprule")
    print("Factor & Entailment & Contradiction & Uncertain & Not Mentioned \\\\")
    print("\\midrule")
    
    factor_names_latex = ['$Q_1$: Shortcuts', '$Q_2$: Expand', '$T_1$: OpenIE', '$T_2$: Enrich']
    
    for factor, name in zip(FACTORS, factor_names_latex):
        vals = [category_effects[cat]['Average'][factor] for cat in CATEGORIES]
        
        formatted = []
        for v in vals:
            if abs(v) > 0.05:
                formatted.append(f"\\textbf{{{v*100:+.1f}}}")
            else:
                formatted.append(f"{v*100:+.1f}")
        
        print(f"{name} & {' & '.join(formatted)} \\\\")
    
    print("\\bottomrule")
    print("\\end{tabular}")
    print("\\end{table}")

generate_latex_category_effects_table()

In [ ]:
def generate_latex_detailed_results_table():
    """Generate detailed LaTeX table with per-category results for one dataset (DocNLI as example)."""
    print("\n% LaTeX Table: Detailed Results with Per-Category Accuracy")
    print("\\begin{table}[t]")
    print("\\centering")
    print("\\caption{Per-category accuracy (\\%) by configuration, averaged across all datasets.}")
    print("\\label{tab:results_detailed}")
    print("\\begin{tabular}{c cccc | c cccc}")
    print("\\toprule")
    print("Run & $Q_1$ & $Q_2$ & $T_1$ & $T_2$ & Overall & Entail & Contra & Uncert & NotMent \\\\")
    print("\\midrule")
    
    for _, row in df.iterrows():
        q1 = '+' if row['Q1_shortcuts'] == 1 else '-'
        q2 = '+' if row['Q2_expand'] == 1 else '-'
        t1 = '+' if row['T1_openie'] == 1 else '-'
        t2 = '+' if row['T2_enrich'] == 1 else '-'
        
        print(f"{int(row['Run'])} & ${q1}$ & ${q2}$ & ${t1}$ & ${t2}$ & "
              f"{row['Average_overall']*100:.1f} & "
              f"{row['Average_entailment']*100:.1f} & "
              f"{row['Average_contradiction']*100:.1f} & "
              f"{row['Average_uncertain']*100:.1f} & "
              f"{row['Average_not_mentioned']*100:.1f} \\\\")
    
    print("\\bottomrule")
    print("\\end{tabular}")
    print("\\end{table}")

generate_latex_detailed_results_table()

---
## 9. Export Data

In [ ]:
# Save complete results to CSV
df.to_csv('results_complete.csv', index=False)

# Save overall effects summary
overall_effects_df = pd.DataFrame(overall_effects)
overall_effects_df['Average'] = pd.Series(avg_overall_effects)
overall_effects_df.to_csv('effects_overall.csv')

# Save category effects summary
for cat in CATEGORIES:
    cat_effects_df = pd.DataFrame(category_effects[cat])
    cat_effects_df.to_csv(f'effects_{cat}.csv')

print("Files saved:")
print("  - results_complete.csv (all raw data)")
print("  - effects_overall.csv (overall accuracy effects)")
for cat in CATEGORIES:
    print(f"  - effects_{cat}.csv ({cat} effects)")
print("\nPlots saved:")
print("  - main_effects_overall.png")
print("  - main_effects_by_category.png")
print("  - heatmap_overall_by_dataset.png")
print("  - heatmap_by_category.png")
print("  - heatmap_complete.png")
print("  - pareto_effects.png")
print("  - interaction_plots.png")
print("  - category_by_config.png")

---
## 10. Baseline Comparison (Optional)

In [ ]:
# ============================================================================
# OPTIONAL: Enter baseline results for comparison
# ============================================================================

baselines = {
    'RAG': {
        'LogiQA2': 0.0,
        'LogicBench': 0.0,
        'DocNLI': 0.0,
        'Alice': 0.0,
    },
    'CoT (GPT-4)': {
        'LogiQA2': 0.0,
        'LogicBench': 0.0,
        'DocNLI': 0.0,
        'Alice': 0.0,
    },
    'Direct Prompting': {
        'LogiQA2': 0.0,
        'LogicBench': 0.0,
        'DocNLI': 0.0,
        'Alice': 0.0,
    }
}

# Convert to 0-1 scale if needed
for method in baselines:
    for ds in DATASETS:
        if baselines[method][ds] > 1:
            baselines[method][ds] /= 100.0

In [ ]:
# Compare best configuration to baselines
if any(baselines[m][DATASETS[0]] > 0 for m in baselines):
    print("\nComparison with Baselines")
    print("="*90)
    
    best_row = df.loc[df['Average_overall'].idxmax()]
    
    print(f"\n{'Method':<25} {'LogiQA2':>10} {'LogicBench':>12} {'DocNLI':>10} {'Alice':>10} {'Average':>10}")
    print("-"*90)
    
    for method, scores in baselines.items():
        avg = np.mean([scores[ds] for ds in DATASETS])
        print(f"{method:<25} {scores['LogiQA2']*100:>10.1f} {scores['LogicBench']*100:>12.1f} "
              f"{scores['DocNLI']*100:>10.1f} {scores['Alice']*100:>10.1f} {avg*100:>10.1f}")
    
    print("-"*90)
    print(f"{'Our Pipeline (best)':<25} {best_row['LogiQA2_overall']*100:>10.1f} {best_row['LogicBench_overall']*100:>12.1f} "
          f"{best_row['DocNLI_overall']*100:>10.1f} {best_row['Alice_overall']*100:>10.1f} {best_row['Average_overall']*100:>10.1f}")
else:
    print("No baseline results entered. Add data above to enable comparison.")

---
## 11. Interpretation Summary

In [ ]:
def generate_interpretation():
    """Generate natural language interpretation of results."""
    print("\n" + "="*80)
    print("INTERPRETATION SUMMARY")
    print("="*80)
    
    factors_info = {
        'Q1_shortcuts': ('Shortcut Detectors', 'use_shortcuts'),
        'Q2_expand': ('Query Expansion + Voting', 'expand_query'),
        'T1_openie': ('OpenIE Triples', 'use_openie'),
        'T2_enrich': ('KB Enrichment', 'use_enrichment_kb')
    }
    
    # Overall main effects
    print("\n1. MAIN EFFECTS ON OVERALL ACCURACY:")
    print("-"*60)
    
    sorted_factors = sorted(FACTORS, key=lambda x: abs(avg_overall_effects[x]), reverse=True)
    
    for factor in sorted_factors:
        name, code = factors_info[factor]
        eff = avg_overall_effects[factor]
        
        direction = "IMPROVES" if eff > 0 else "DECREASES"
        magnitude = abs(eff)
        
        if magnitude > 0.10:
            strength = "strongly"
        elif magnitude > 0.05:
            strength = "moderately"
        else:
            strength = "weakly"
        
        print(f"  {name} ({code}): {direction} accuracy by {magnitude:.1%} {strength}")
    
    # Category-specific insights
    print("\n2. CATEGORY-SPECIFIC INSIGHTS:")
    print("-"*60)
    
    for cat in CATEGORIES:
        best_factor = max(FACTORS, key=lambda x: category_effects[cat]['Average'][x])
        worst_factor = min(FACTORS, key=lambda x: category_effects[cat]['Average'][x])
        
        best_eff = category_effects[cat]['Average'][best_factor]
        worst_eff = category_effects[cat]['Average'][worst_factor]
        
        print(f"\n  {cat.upper()}:")
        if abs(best_eff) > 0.03:
            print(f"    Most helpful: {factors_info[best_factor][0]} ({best_eff:+.1%})")
        if abs(worst_eff) > 0.03:
            print(f"    Most harmful: {factors_info[worst_factor][0]} ({worst_eff:+.1%})")
    
    # Best configuration
    print("\n3. RECOMMENDED CONFIGURATION:")
    print("-"*60)
    
    best_config = df.loc[df['Average_overall'].idxmax()]
    print(f"  Best overall: Run {int(best_config['Run'])} (Accuracy: {best_config['Average_overall']:.1%})")
    for factor in FACTORS:
        name, code = factors_info[factor]
        setting = 'ON' if best_config[factor] == 1 else 'OFF'
        print(f"    - {code}: {setting}")

generate_interpretation()

---

## Summary

This notebook provides comprehensive analysis of your $2^{4-1}_{IV}$ fractional factorial experiment:

**Input**:
- Overall accuracy for each run × dataset
- Per-category accuracy (entailment, contradiction, uncertain, not_mentioned)

**Analysis**:
1. Main effects and interactions for overall accuracy
2. Main effects for each response category
3. Bootstrap significance testing
4. Multiple visualizations (heatmaps, bar charts, interaction plots)

**Output**:
- CSV files with all effects
- Publication-ready LaTeX tables
- High-resolution plots
- Automated interpretation

**Next steps**:
1. Fill in your experimental results in Section 2
2. Run all cells
3. Copy LaTeX tables from Section 8
4. Send me the outputs and I'll write your Results section